# Задача 2. Прогнозирование стоимости на N лет вперёд — Dynamic Ridge Regression

В ноутбуке отдельно подбираются гиперпараметры **только Dynamic Ridge Regression**. Остальные алгоритмы не сравниваются.

Сохраняется исходная схема прогноза: фиксированный CatBoost оценивает базовую стоимость недвижимости, а Dynamic Ridge рассчитывает временную поправку к логарифму цены за квадратный метр. Динамичность обеспечивается скользящим окном исторических наблюдений и экспоненциальным уменьшением веса более старых сделок. На каждом временном fold модель заново обучается только на данных, доступных на соответствующий год.

Гиперпараметры подбираются по временной кросс-валидации на периодах до 2026 года. Итоговая проверка выполняется на backtest 2026 года. Таблица HTML содержит одну строку — результат Dynamic Ridge Regression.


## 1. Библиотеки, пути и общие параметры

Подключаются существующие функции очистки и подготовки данных проекта. EDA не выполняется. Настройки общего CatBoost базовой цены фиксированы, чтобы сравнение отражало качество пяти моделей временной поправки.


In [1]:
from pathlib import Path
import sys
import time
import warnings
import inspect
from itertools import product

import numpy as np
import pandas as pd
from IPython.display import display, HTML

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.exceptions import ConvergenceWarning

from catboost import CatBoostRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT = Path.cwd()
CSV_NAME = "СДЕЛКИ_Мос_2023-2026.csv"
while not (ROOT / CSV_NAME).exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / CSV_NAME).exists():
    raise FileNotFoundError(f"Не найден файл данных: {CSV_NAME}")

sys.path.insert(0, str(ROOT / "web_app"))
import model as app_model

DATA_PATH = ROOT / CSV_NAME
RANDOM_STATE = 42
FEATURE_COLS = app_model.FEATURE_COLS
CAT_COLS = app_model.CATEGORICAL_COLS
NUM_COLS = app_model.NUMERIC_COLS
BASE_YEAR = 2026
BACKTEST_MIN_DEALS = 1000

# Общая базовая модель CatBoost — фиксирована и не сравнивается с кандидатами.
BASE_CAT_PARAMS = {
    "iterations": 555,
    "depth": 10,
    "learning_rate": 0.056794,
    "l2_leaf_reg": 2.0917992640459695,
    "random_strength": 0.5598053604013576,
    "loss_function": "RMSE",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "thread_count": 3,
    "allow_writing_files": False,
}

print("Данные:", DATA_PATH)
print("Базовый год backtest:", BASE_YEAR)
print("Единственная прогнозная модель: Dynamic Ridge Regression")
print("Признаки:", FEATURE_COLS)


Данные: C:\Users\AnnaA\Desktop\Prognosis_system\СДЕЛКИ_Мос_2023-2026.csv
Базовый год backtest: 2026
Единственная прогнозная модель: Dynamic Ridge Regression
Признаки: ['region', 'district', 'street', 'wall_material_code', 'year_build', 'floor', 'area', 'period_year']


## 2. Подготовка данных и временных fold

Используется существующая очистка данных и последовательная фильтрация выбросов. Для каждого fold границы фильтра вычисляются только по train.

В подборе гиперпараметров проверяются исторические прогнозы, где год обучения строго предшествует целевому году, а целевой год меньше 2026. 2026 год остаётся отдельной итоговой проверкой.


In [2]:
df_raw = app_model.read_csv_robust(DATA_PATH)
df_raw.columns = df_raw.columns.astype(str).str.strip().str.lower()
clean_df = app_model.clean_data(df_raw)


def prepare_backtest_pair(train_raw, test_raw):
    """Применяет фильтры проекта; границы выбросов оцениваются только на train."""
    train = train_raw.copy()
    test = test_raw.copy()
    train["price_per_sqm"] = train["price"] / train["area"]
    test["price_per_sqm"] = test["price"] / test["area"]

    for column in ("price", "price_per_sqm"):
        mean = float(train[column].mean())
        std = float(train[column].std())
        if not np.isfinite(std) or std <= 0:
            continue
        lower, upper = mean - 3 * std, mean + 3 * std
        train = train[train[column].between(lower, upper)].copy()
        test = test[test[column].between(lower, upper)].copy()

    def finalize(frame):
        frame = frame.copy()
        for column in CAT_COLS:
            frame[column] = frame[column].map(app_model.normalize_text)
        frame["wall_material_code"] = (
            frame["wall_material_code"].astype(str).str.replace(r"\.0+$", "", regex=True)
        )
        for column in list(NUM_COLS) + ["price", "period_year"]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(subset=list(NUM_COLS) + ["price", "period_year"]).copy()
        frame = frame[
            frame["period_year"].between(2023, 2026)
            & frame["year_build"].between(1800, 2027)
            & (frame["year_build"] <= frame["period_year"] + 1)
            & frame["floor"].between(1, 100)
            & frame["area"].between(25, 400)
            & frame["price"].between(7_000_000, 1_000_000_000)
        ].copy()
        frame["price_per_sqm"] = frame["price"] / frame["area"]
        frame = frame[frame["price_per_sqm"].between(50_000, 15_000_000)].copy()
        frame["period_year"] = frame["period_year"].astype(int)
        return frame.sort_values("period_start_date").reset_index(drop=True)

    return finalize(train), finalize(test)


available_years = sorted(clean_df["period_year"].dropna().astype(int).unique())
cv_folds = []

for origin_year in available_years:
    if origin_year >= BASE_YEAR:
        continue
    for target_year in available_years:
        if origin_year < target_year < BASE_YEAR:
            raw_train = clean_df[clean_df["period_year"] <= origin_year].copy()
            raw_test = clean_df[clean_df["period_year"] == target_year].copy()
            if len(raw_train) < 30 or len(raw_test) < BACKTEST_MIN_DEALS:
                continue

            train_fold, test_fold = prepare_backtest_pair(raw_train, raw_test)
            if len(train_fold) >= 30 and len(test_fold) >= BACKTEST_MIN_DEALS:
                cv_folds.append((origin_year, target_year, train_fold, test_fold))

if not cv_folds:
    raise ValueError("Для временной кросс-валидации недостаточно исторических fold.")

print("Fold для подбора:", [(origin, target, len(test)) for origin, target, _, test in cv_folds])
print("Данные 2026 года будут использованы только в итоговом backtest.")


Fold для подбора: [(np.int64(2023), np.int64(2024), 14682), (np.int64(2023), np.int64(2025), 19305), (np.int64(2024), np.int64(2025), 23747)]
Данные 2026 года будут использованы только в итоговом backtest.


## 3. Dynamic Ridge Regression

Используется только одна прогнозная модель — Dynamic Ridge Regression. В отличие от статической Ridge, она на каждом временном fold переобучается на скользящем окне последних лет и использует экспоненциальные веса: чем старше наблюдение, тем меньше его вклад.

Подбираются три гиперпараметра: `alpha` (сила L2-регуляризации), `window_years` (ширина исторического окна) и `half_life_years` (период, за который вес наблюдения уменьшается вдвое). Базовая стоимость по-прежнему рассчитывается общей фиксированной моделью CatBoost.


In [3]:
MODEL_DYNAMIC_RIDGE = "Dynamic Ridge Regression"


def catboost_frame(frame):
    result = frame[FEATURE_COLS].copy()
    for column in CAT_COLS:
        result[column] = result[column].astype(str)
    return result


def fit_base_catboost(train):
    """Фиксированная базовая модель CatBoost — не подбирается."""
    model = CatBoostRegressor(**BASE_CAT_PARAMS)
    X = catboost_frame(train)
    y = np.log(train["price"].to_numpy(float) / train["area"].to_numpy(float))
    model.fit(X, y, cat_features=CAT_COLS)
    return model


def predict_base_log(model, X):
    return model.predict(catboost_frame(X))


def make_candidate_matrix(frame, support, vectorizer=None, scaler=None, fit=False):
    records = [
        app_model.record_features(
            row, support, with_time=True, trend_scale=1.0
        )
        for row in frame[FEATURE_COLS].to_dict("records")
    ]
    if fit:
        vectorizer = DictVectorizer(sparse=True)
        matrix = vectorizer.fit_transform(records)
        scaler = StandardScaler(with_mean=False)
        matrix = scaler.fit_transform(matrix)
        return matrix, vectorizer, scaler
    return scaler.transform(vectorizer.transform(records))


def make_dynamic_weights(train, half_life_years):
    """Экспоненциальное затухание веса: возраст сделки измеряется в годах."""
    latest_year = int(train["period_year"].max())
    age = latest_year - train["period_year"].to_numpy(float)
    return np.power(0.5, age / float(half_life_years))


def fit_dynamic_ridge(params, train):
    """
    Dynamic Ridge:
    1) оставляет скользящее окно последних window_years лет;
    2) сильнее взвешивает более свежие наблюдения;
    3) заново оценивает Ridge на каждом временном fold.
    """
    latest_year = int(train["period_year"].max())
    first_year = latest_year - int(params["window_years"]) + 1
    dynamic_train = train[train["period_year"] >= first_year].copy()
    if len(dynamic_train) < 30:
        raise ValueError(
            f"В окне Dynamic Ridge осталось только {len(dynamic_train)} наблюдений."
        )

    y = np.log(
        dynamic_train["price"].to_numpy(float)
        / dynamic_train["area"].to_numpy(float)
    )
    weights = make_dynamic_weights(
        dynamic_train, params["half_life_years"]
    )

    support = app_model.make_support(dynamic_train)
    matrix, vectorizer, scaler = make_candidate_matrix(
        dynamic_train, support, fit=True
    )

    estimator = Ridge(
        alpha=float(params["alpha"]),
        solver="lsqr",
        fit_intercept=True,
        tol=1e-4,
        max_iter=5000,
    )
    estimator.fit(matrix, y, sample_weight=weights)

    return {
        "name": MODEL_DYNAMIC_RIDGE,
        "params": params.copy(),
        "model": estimator,
        "support": support,
        "vectorizer": vectorizer,
        "scaler": scaler,
        "window_start_year": first_year,
        "window_end_year": latest_year,
    }


def predict_dynamic_ridge(fitted, X):
    matrix = make_candidate_matrix(
        X,
        fitted["support"],
        vectorizer=fitted["vectorizer"],
        scaler=fitted["scaler"],
        fit=False,
    )
    return fitted["model"].predict(matrix)


def forecast_log_price(base_model, trend_model, test):
    """Базовая цена CatBoost × временная поправка Dynamic Ridge."""
    origin_year = int(trend_model["origin_year"])
    target_year = int(trend_model["target_year"])

    X_origin = test[FEATURE_COLS].copy()
    X_target = test[FEATURE_COLS].copy()
    X_origin["period_year"] = origin_year
    X_target["period_year"] = target_year

    base_log = predict_base_log(base_model, X_origin)
    trend_origin = predict_dynamic_ridge(trend_model, X_origin)
    trend_target = predict_dynamic_ridge(trend_model, X_target)
    return base_log + trend_target - trend_origin


# Подбор только для Dynamic Ridge Regression.
# window_years — длина скользящего окна; half_life_years — период
# уменьшения веса старых наблюдений вдвое; alpha — L2-регуляризация.
PARAMETER_GRIDS = {
    MODEL_DYNAMIC_RIDGE: [
        {
            "alpha": alpha,
            "window_years": window_years,
            "half_life_years": half_life_years,
        }
        for alpha, window_years, half_life_years in product(
            [0.1, 1.0, 10.0, 100.0, 500.0],
            [1, 2, 3, 4],
            [0.5, 1.0, 2.0, 4.0],
        )
    ]
}


## 4. Подбор гиперпараметров Dynamic Ridge

Все комбинации проверяются на одинаковых исторических временных fold. Критерий выбора — минимальная средняя `CV MAE (log)` итоговой прогнозной цены за квадратный метр. Данные 2026 года не участвуют в подборе.


In [4]:
# Кэшируем общий CatBoost для каждого временного CV-fold.
if "base_models_by_fold" not in globals():
    base_models_by_fold = {}
    for origin_year, target_year, train_fold, _ in cv_folds:
        base_models_by_fold[(origin_year, target_year)] = fit_base_catboost(train_fold)

# Результаты конфигураций сохраняются в памяти; повторный запуск продолжает подбор.
if "cv_scores" not in globals() or set(cv_scores) != set(PARAMETER_GRIDS):
    cv_scores = {name: [] for name in PARAMETER_GRIDS}
if "best_params" not in globals() or set(best_params) != set(PARAMETER_GRIDS):
    best_params = {}
if "tuning_times" not in globals() or set(tuning_times) != set(PARAMETER_GRIDS):
    tuning_times = {}

for name, grid in PARAMETER_GRIDS.items():
    scores = cv_scores.setdefault(name, [])
    if len(scores) >= len(grid) and name in best_params:
        print(f"{name}: подбор уже завершён; параметры={best_params[name]}")
        continue

    started = time.perf_counter()
    for params in grid[len(scores):]:
        fold_errors = []
        for origin_year, target_year, train_fold, test_fold in cv_folds:
            base_model = base_models_by_fold[(origin_year, target_year)]
            candidate = fit_dynamic_ridge(params, train_fold)
            candidate["origin_year"] = origin_year
            candidate["target_year"] = target_year

            predicted_log_ppsm = forecast_log_price(
                base_model, candidate, test_fold
            )
            actual_log_ppsm = np.log(
                test_fold["price"].to_numpy(float)
                / test_fold["area"].to_numpy(float)
            )
            fold_errors.extend(
                np.abs(actual_log_ppsm - predicted_log_ppsm).tolist()
            )

        scores.append((float(np.mean(fold_errors)), params.copy()))

    best_score, best_config = min(scores, key=lambda item: item[0])
    best_params[name] = best_config.copy()
    tuning_times[name] = tuning_times.get(name, 0.0) + (
        time.perf_counter() - started
    )
    print(
        f"{name}: CV MAE(log)={best_score:.6f}; "
        f"лучшие параметры={best_config}; "
        f"время подбора={tuning_times[name]:.1f} с"
    )

cv_best_log_mae = {
    name: min(score for score, _ in cv_scores[name])
    for name in PARAMETER_GRIDS
}
print("\nПодбор гиперпараметров Dynamic Ridge завершён.")


Dynamic Ridge Regression: CV MAE(log)=0.213450; лучшие параметры={'alpha': 500.0, 'window_years': 1, 'half_life_years': 0.5}; время подбора=563.2 с

Подбор гиперпараметров Dynamic Ridge завершён.


## 5. Итоговый backtest 2026 года

Dynamic Ridge с лучшими гиперпараметрами обучается на исторической выборке и прогнозирует сделки 2026 года. `Time, s` — время одного финального обучения Dynamic Ridge на лучших параметрах; время подбора и обучение CatBoost в этот показатель не входят.


In [5]:
train_raw = clean_df[clean_df["period_year"] < BASE_YEAR].copy()
test_raw = clean_df[clean_df["period_year"] == BASE_YEAR].copy()

train_2026, test_2026 = prepare_backtest_pair(train_raw, test_raw)
if len(train_2026) < 30 or len(test_2026) < BACKTEST_MIN_DEALS:
    raise ValueError(
        "Для backtest 2026 года недостаточно подходящих обучающих или тестовых сделок."
    )

# CatBoost — фиксированная базовая оценка, не входит во время обучения Dynamic Ridge.
base_model_2026 = fit_base_catboost(train_2026)

best_dynamic_params = best_params[MODEL_DYNAMIC_RIDGE]
fit_started = time.perf_counter()
dynamic_model = fit_dynamic_ridge(best_dynamic_params, train_2026)
dynamic_fit_time = time.perf_counter() - fit_started

dynamic_model["origin_year"] = BASE_YEAR - 1
dynamic_model["target_year"] = BASE_YEAR

predicted_log_ppsm = forecast_log_price(
    base_model_2026, dynamic_model, test_2026
)
predicted_price = (
    np.exp(np.clip(predicted_log_ppsm, -20, 25))
    * test_2026["area"].to_numpy(float)
)
actual_price = test_2026["price"].to_numpy(float)

mae = mean_absolute_error(actual_price, predicted_price) / 1e6
rmse = np.sqrt(mean_squared_error(actual_price, predicted_price)) / 1e6
mape = (
    np.mean(
        np.abs(actual_price - predicted_price)
        / np.maximum(actual_price, 1.0)
    )
    * 100
)
r2 = r2_score(actual_price, predicted_price)

result_table = pd.DataFrame([{
    "Algorithm": MODEL_DYNAMIC_RIDGE,
    "MAE, million RUB": mae,
    "RMSE, million RUB": rmse,
    "MAPE, %": mape,
    "R²": r2,
    "CV MAE (log)": cv_best_log_mae[MODEL_DYNAMIC_RIDGE],
    "Time, s": dynamic_fit_time,
    "Accuracy, %": 100.0 - mape,
}])

TABLE_COLUMNS = [
    "Algorithm",
    "MAE, million RUB",
    "RMSE, million RUB",
    "MAPE, %",
    "R²",
    "CV MAE (log)",
    "Time, s",
    "Accuracy, %",
]

display(HTML(
    result_table[TABLE_COLUMNS].to_html(
        index=False,
        escape=False,
        float_format=lambda value: f"{value:.4f}",
        border=0,
        classes="table table-striped",
    )
))

print(f"Обучающих сделок до 2025 года включительно: {len(train_2026):,}")
print(f"Тестовых сделок 2026 года: {len(test_2026):,}")


Algorithm,"MAE, million RUB","RMSE, million RUB","MAPE, %",R²,CV MAE (log),"Time, s","Accuracy, %"
Dynamic Ridge Regression,7.1635,10.1193,27.9677,0.4825,0.2135,1.3056,72.0323


Обучающих сделок до 2025 года включительно: 45,017
Тестовых сделок 2026 года: 15,489


## 6. Итог и лучшие гиперпараметры

В таблице приведены метрики единственной проверяемой модели. Лучшие гиперпараметры определяются исключительно по временной кросс-валидации, а MAE итогового backtest используется для оценки качества на отложенном 2026 году.


In [6]:
print("=" * 72)
print("DYNAMIC RIDGE REGRESSION — ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ")
print("=" * 72)
print(f"CV MAE (log): {cv_best_log_mae[MODEL_DYNAMIC_RIDGE]:.6f}")
for parameter, value in best_params[MODEL_DYNAMIC_RIDGE].items():
    print(f"{parameter}: {value}")
print("\nИтоговый backtest 2026:")
print(f"MAE: {mae:.4f} million RUB")
print(f"RMSE: {rmse:.4f} million RUB")
print(f"MAPE: {mape:.4f}%")
print(f"R²: {r2:.4f}")
print(f"Accuracy: {100.0 - mape:.4f}%")
print(f"Time (one final model fit): {dynamic_fit_time:.4f} s")


DYNAMIC RIDGE REGRESSION — ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ
CV MAE (log): 0.213450
alpha: 500.0
window_years: 1
half_life_years: 0.5

Итоговый backtest 2026:
MAE: 7.1635 million RUB
RMSE: 10.1193 million RUB
MAPE: 27.9677%
R²: 0.4825
Accuracy: 72.0323%
Time (one final model fit): 1.3056 s
